# VUEMesh films: Qwen-Image stills, then LTX image to video

Three films, 28 shots. Each shot has an image prompt (Qwen-Image makes the still) and a motion prompt (LTX turns the still into a clip), in `shots.json`. Run the cells in order; anything already made is skipped.

**Where's ComfyUI, from inside this Jupyter container?** `127.0.0.1` here is the container itself, not ComfyUI:

| ComfyUI runs… | Address | Also |
|---|---|---|
| in another container on the same Docker network (e.g. the same compose file) | `http://comfyui:8188` (its service name) | |
| on the host machine | `http://host.docker.internal:8188` | start ComfyUI with `--listen`; on Linux, give the Jupyter service `extra_hosts: ["host.docker.internal:host-gateway"]` |
| on another computer | `http://192.168.x.x:8188` | start ComfyUI with `--listen` |

Set it below, or as `COMFY_URL` in the container's environment.

In [ ]:
import os, sys, json, glob
sys.path.insert(0, os.path.abspath('.'))     # this folder (comfy_pipeline.py, shots.json, your workflows)
import comfy_pipeline as cp

SERVER = os.environ.get('COMFY_URL', 'http://host.docker.internal:8188')
info = cp.check(server=SERVER)   # (the GPU it found, and its free memory)

## Your workflows

In ComfyUI, export your Qwen-Image workflow and your LTX image-to-video workflow with **Workflow > Export (API)**, and put them in this folder as `image_workflow.json` and `video_workflow.json`. This shows which nodes the script will set; if one's wrong or missing, rename that node in ComfyUI (right-click > Title) to the name it suggests, and export again.

In [ ]:
cp.inspect('image_workflow.json', server=SERVER)
print()
cp.inspect('video_workflow.json', server=SERVER)

## 1. Stills

`ONLY` can be a film (`'wedding'`, `'concert'`, `'arena'`), one shot (`'wedding/05-dance-wide'`), or `None` for everything. `FORCE = True` makes them again with new seeds.

In [ ]:
ONLY = 'wedding'
VARIANTS = 4
FORCE = False
cp.stills(only=ONLY, variants=VARIANTS, force=FORCE, server=SERVER)

## 2. Pick a still for each shot

Click the number under the still you want; it's saved to `picks.json` straight away. (Shots you don't pick use still 1.)

In [ ]:
from IPython.display import display, HTML
OUT = cp.load_settings(server=SERVER)['out']
PICKS = os.path.join(OUT, 'picks.json')
picks = json.load(open(PICKS)) if os.path.exists(PICKS) else {}

def save_picks():
    os.makedirs(OUT, exist_ok=True)
    with open(PICKS, 'w') as f:
        json.dump(picks, f, indent=2)

def thumb(path, width=320):
    # a small JPEG, so the notebook stays light (the full still is on disk)
    try:
        from PIL import Image
        import io
        im = Image.open(path).convert('RGB')
        im.thumbnail((width, width))
        buf = io.BytesIO()
        im.save(buf, 'JPEG', quality=82)
        return buf.getvalue(), 'jpeg'
    except ImportError:
        return open(path, 'rb').read(), 'png'

def still_numbers(film, shot):
    files = glob.glob(os.path.join(OUT, film['id'], shot['id'], 'still_v*.png'))
    return sorted((int(p.rsplit('_v', 1)[1][:-4]), p) for p in files)

try:
    import ipywidgets as w
    rows = []
    for film, shot, key in cp.shots(server=SERVER):
        found = still_numbers(film, shot)
        if not found:
            continue
        nums = [n for n, _ in found]
        imgs = []
        for n, p in found:
            data, fmt = thumb(p)
            imgs.append(w.VBox([w.Image(value=data, format=fmt, width=280), w.HTML(f'<center>{n}</center>')]))
        pick = w.ToggleButtons(options=nums, value=picks.get(key) if picks.get(key) in nums else nums[0],
                               description='Use', style={'button_width': '44px'})
        def chosen(ch, key=key):
            picks[key] = ch['new']
            save_picks()
        pick.observe(chosen, names='value')
        note = f"<b>{key}</b> · {shot.get('seconds', 4)} s<br><small>{shot['motion']}</small>"
        if shot.get('overlay'):
            note += f"<br><small style='color:#b8720b'>Overlay: {shot['overlay']}</small>"
        rows.append(w.VBox([w.HTML(note), w.HBox(imgs), pick], layout=w.Layout(margin='0 0 18px 0')))
    display(w.VBox(rows) if rows else w.HTML('No stills yet: run the stills cell first.'))
except ImportError:
    import base64
    for film, shot, key in cp.shots(server=SERVER):
        found = still_numbers(film, shot)
        if found:
            cells = ''.join(f"<figure style='display:inline-block;margin:4px'><img src='data:image/{fmt};base64,{base64.b64encode(d).decode()}' width=240><figcaption>{n}</figcaption></figure>"
                            for n, p in found for d, fmt in [thumb(p)])
            display(HTML(f'<b>{key}</b><br>{cells}'))
    print('No ipywidgets here: choose with  picks["wedding/05-dance-wide"] = 2; save_picks()')

## 3. Clips

In [ ]:
cp.videos(only=ONLY, force=FORCE, server=SERVER)

## Watch them

In [ ]:
from IPython.display import Video
for film, shot, key in cp.shots(server=SERVER):
    for clip in sorted(glob.glob(os.path.join(OUT, film['id'], shot['id'], 'clip_v*'))):
        display(HTML(f'<b>{key}</b> (still {clip.rsplit("_v", 1)[1].split(".")[0]})'))
        display(Video(os.path.relpath(clip), width=560))

## Music (ACE-Step)

An instrumental bed for each film, from your ACE-Step workflow (export it with **Workflow > Export (API)** as `music_workflow.json`). Each film's style is its `music.tags` in `shots.json`; the length follows its cut. Three takes per film; pick one below, and the finished film uses it. (To use your own recording instead, put it in `renders/music/<film>/` as `music_v1.wav` and pick 1.)

In [ ]:
import importlib; importlib.reload(cp)
cp.inspect('music_workflow.json', server=SERVER)
# force=True makes the takes again (new seeds), replacing any already made
cp.music(only=None, variants=3, force=False, server=SERVER)

In [ ]:
# listen, and pick a take for each film (saved as you click)
from IPython.display import Audio
picks = json.load(open(PICKS)) if os.path.exists(PICKS) else {}
try:
    import ipywidgets as w
    rows = []
    for film in cp.load_json(cp.load_settings(server=SERVER)['shots'])['films']:
        folder = os.path.join(OUT, 'music', film['id'])
        takes = sorted(f for f in os.listdir(folder) if f.startswith('music_v')) if os.path.isdir(folder) else []
        if not takes:
            continue
        nums = [int(t[7:].split('.')[0]) for t in takes]
        players = [w.VBox([w.HTML(f'<b>{n}</b>'), w.Output()]) for n in nums]
        for box, t in zip(players, takes):
            with box.children[1]:
                display(Audio(os.path.join(folder, t)))
        key = f"music/{film['id']}"
        pick = w.ToggleButtons(options=nums, value=picks.get(key) if picks.get(key) in nums else nums[0], description='Use', style={'button_width': '44px'})
        def chosen(ch, key=key):
            picks[key] = ch['new']
            save_picks()
        pick.observe(chosen, names='value')
        rows.append(w.VBox([w.HTML(f"<b>{film['title']}</b> <small>{film['music']['tags']}</small>"), w.HBox(players), pick]))
    display(w.VBox(rows) if rows else w.HTML('No music yet: run the cell above.'))
except ImportError:
    for film in cp.load_json(cp.load_settings(server=SERVER)['shots'])['films']:
        folder = os.path.join(OUT, 'music', film['id'])
        for t in sorted(os.listdir(folder)) if os.path.isdir(folder) else []:
            display(HTML(f"<b>{film['id']}</b> take {t[7:].split('.')[0]}")); display(Audio(os.path.join(folder, t)))
    print('No ipywidgets here: choose with  picks["music/wedding"] = 2; save_picks()')

## 4. The finished films

Each film is cut as its `edit` in `shots.json` says: the part of each clip used, its speed, and how it comes in (cut, dissolve, through black, or a flash through white), with its title, a colour grade so the shots match, and an end card. It uses the clip made from the still you picked for each shot. Out come `renders/final/wedding.mp4`, `concert.mp4`, `arena.mp4` and `reel.mp4` (all three).

Each film uses its picked music take (above); `MUSIC` overrides that with a file for every film, or one each, e.g. `{'wedding': 'song.mp3'}`. With no music at all, the film is silent. Needs ffmpeg (most Jupyter images have it; otherwise `pip install imageio-ffmpeg`).

In [ ]:
import importlib; importlib.reload(cp)   # (picks up a newer comfy_pipeline.py without restarting)
MUSIC = None   # (None: each film's picked ACE-Step take, if there is one; or a file to use instead)
films = cp.cut(music=MUSIC, server=SERVER)
for path in films:
    display(HTML(f'<b>{os.path.basename(path)}</b>'))
    display(Video(os.path.relpath(path), width=720))

Every clip in one folder (`renders/final/clips/`), with a `list.txt` saying which still each came from and which shots still need their on-screen overlay:

In [ ]:
cp.collect(server=SERVER)